# Extract Items to CSV

This notebook extracts all items from Anno 117 into a CSV file with the following columns:
- guid
- name (English)
- rarity
- trade_price
- targets (comma-separated English names)
- buffs (descriptions of all buff attributes)
- boost_condition (for ItemWithBoost: condition when boost activates)
- boost_buffs (for ItemWithBoost: boosted buff attributes)
- source (where the item can be obtained: traders, research, quests, expeditions)

In [94]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import Asset, AssetCache
from assetextractor.parsing.core.templates import Template
from assetextractor.parsing.core.attributes import ListAttribute, PrimitiveAttribute, UpgradeAttribute

import csv
from pathlib import Path
import typing as t
import pandas as pd

In [95]:
# Load asset cache
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

print(f"Loaded assets successfully")



Loaded assets successfully


In [115]:
def flatten_pool(pool: Asset | None) -> list[int]:
    """Recursively flatten an AssetPool into individual asset GUIDs."""
    if pool is None:
        return []

    if "AssetPool" not in pool.template.name:
        return [pool.guid]

    res = []
    for entry in pool.AssetPool.AssetList:
        if entry.Asset():  # Last item can sometimes be None
            res += flatten_pool(entry.Asset())
    return res


def get_english_name(asset: Asset) -> str:
    """Get the English name of an asset from localization."""
    if asset.text is not None and "english" in asset.text.values:
        return asset.text.values["english"]
    # Fallback to internal name
    try:
        name = asset.find("Standard.Name")()
        return name if name else f"Asset_{asset.guid}"
    except:
        return f"Asset_{asset.guid}"


def get_target_names(target_guids: list[int]) -> str:
    """Convert list of target GUIDs to comma-separated English names."""
    names = []
    for guid in target_guids:
        try:
            target_asset = assets[guid]
            if target_asset:
                names.append(get_english_name(target_asset))
        except:
            names.append(f"Unknown_{guid}")
    return ", ".join(names) if names else ""


# === Item Source Tracking ===

def build_reward_pool_index(assets: AssetCache) -> dict[int, set[int]]:
    """
    Build an index mapping item GUIDs to RewardPool GUIDs that contain them.
    
    Returns: {item_guid: {reward_pool_guid1, reward_pool_guid2, ...}}
    """
    item_to_pools = {}
    
    # Find all RewardPool templates
    for template_name in ["RewardPool", "RegionRewardPool"]:
        if template_name not in assets.templates:
            continue
            
        for pool_asset in assets.templates[template_name].assets:
            try:
                items_pool = pool_asset.RewardPool.ItemsPool
                if items_pool:
                    for item_entry in items_pool:
                        try:
                            item_link = item_entry.ItemLink()
                            if item_link:
                                # This could be an item or another RewardPool
                                # We'll resolve the chain later
                                if item_link.guid not in item_to_pools:
                                    item_to_pools[item_link.guid] = set()
                                item_to_pools[item_link.guid].add(pool_asset.guid)
                        except:
                            pass
            except:
                pass
    
    return item_to_pools

item_to_pools = build_reward_pool_index(assets)
print(f"Indexed {len(item_to_pools)} items in reward pools")

def resolve_reward_pool_chain(item_guid: int, item_to_pools: dict[int, set[int]], visited: set[int] = None) -> set[int]:
    """
    Recursively resolve the chain of RewardPools that eventually contain this item.
    
    Returns all RewardPool GUIDs that directly or indirectly contain this item.
    """
    if visited is None:
        visited = set()
    
    if item_guid in visited:
        return set()
    
    visited.add(item_guid)
    result = set()
    
    if item_guid in item_to_pools:
        for pool_guid in item_to_pools[item_guid]:
            result.add(pool_guid)
            # Recursively find pools that contain this pool
            result.update(resolve_reward_pool_chain(pool_guid, item_to_pools, visited))
    
    return result


def find_item_sources(item_guid: int, item_to_pools: dict[int, set[int]], assets: AssetCache) -> list[str]:
    """
    Find all sources where this item can be obtained.
    
    Returns a list of source descriptions (e.g., "Trader: Valeria", "Research: Advanced Logistics")
    """
    sources = []
    
    # Get all reward pools that contain this item (directly or indirectly)
    all_pools = resolve_reward_pool_chain(item_guid, item_to_pools)
    
    # For each pool, find what references it
    for pool_guid in all_pools:
        try:
            pool_asset = assets[pool_guid]
            if not pool_asset:
                continue
                
            # Check "Referenced by" - we need to iterate through all assets to find references
            # This is expensive, so we'll search specific templates that commonly reference reward pools
            
            # 1. Check Traders (Participant 3rdParty)
            if "Participant 3rdParty" in assets.templates:
                for trader in assets.templates["Participant 3rdParty"].assets:
                    try:
                        offered_items = trader.find("Trader.OfferedItems")
                        if offered_items and offered_items() and offered_items().guid == pool_guid:
                            trader_name = get_english_name(trader)
                            sources.append(f"Trader: {trader_name}")
                    except:
                        pass
            
            # 2. Check Researches
            if pool_guid == 79669:
                sources.append("Research: Specialist Economy")
            if pool_guid == 79670:
                sources.append("Research: Specialist Civic")
            if pool_guid == 79671:
                sources.append("Research: Specialist Military")

            
            
            # 4. Check Expeditions
            if "Expedition" in assets.templates:
                for expedition in assets.templates["Expedition"].assets:
                    try:
                        expedition_rewards = expedition.find("Expedition.ExpeditionReward")
                        if expedition_rewards:
                            for reward_entry in expedition_rewards:
                                try:
                                    reward_pool = reward_entry.RewardPool()
                                    if reward_pool and reward_pool.guid == pool_guid:
                                        exp_name = get_english_name(expedition)
                                        sources.append(f"Expedition: {exp_name}")
                                except:
                                    pass
                    except:
                        pass
                        
        except:
            pass
    
    # Remove duplicates while preserving order
    seen = set()
    unique_sources = []
    for source in sources:
        if source not in seen:
            seen.add(source)
            unique_sources.append(source)
    
    return unique_sources


# === Boost Condition Extractors ===

def is_default_value(value):
    """Check if a value is a default value that should be skipped."""
    if value is None:
        return True
    if isinstance(value, bool) and value == False:
        return True
    if isinstance(value, (int, float)) and value == 0:
        return True
    if isinstance(value, str) and value == "":
        return True
    if isinstance(value, (list, tuple, set, dict)) and len(value) == 0:
        return True
    return False


def log_attribute_recursively(obj, indent=0, visited=None):
    """Recursively log non-default attributes from an object."""
    if visited is None:
        visited = set()
    
    # Avoid infinite recursion
    obj_id = id(obj)
    if obj_id in visited:
        return
    visited.add(obj_id)
    
    prefix = "  " * indent
    
    # Skip internal attributes
    internal_attrs = {
        'cache', 'parent', 'node', 'meta', 'inherited', 'attributes', 
        'properties', 'name', 'template', '_value_list', '_attribute_name'
    }
    
    try:
        if hasattr(obj, '__dict__'):
            for key, value in obj.__dict__.items():
                # Skip internal attributes
                if key.startswith('_') or key in internal_attrs:
                    continue
                
                try:
                    # Try calling if it's a method
                    if callable(value):
                        result = value()
                        
                        # Skip default values
                        if is_default_value(result):
                            continue
                        
                        # If result is an Asset, print its name
                        if hasattr(result, 'guid') and hasattr(result, 'text'):
                            try:
                                name = get_english_name(result)
                                print(f"{prefix}{key}: {name} (GUID: {result.guid})")
                            except:
                                print(f"{prefix}{key}: Asset {result.guid}")
                        # If result is a simple type, print it
                        elif isinstance(result, (str, int, float, bool)):
                            print(f"{prefix}{key}: {result}")
                        # If it's a dict-like object, recurse
                        elif isinstance(result, dict):
                            print(f"{prefix}{key}:")
                            for k, v in result.items():
                                if not is_default_value(v):
                                    print(f"{prefix}  {k}: {v}")
                        # Otherwise recurse into it
                        else:
                            print(f"{prefix}{key}:")
                            log_attribute_recursively(result, indent + 1, visited)
                    else:
                        # Non-callable attribute
                        if not is_default_value(value):
                            if isinstance(value, (str, int, float, bool)):
                                print(f"{prefix}{key}: {value}")
                            else:
                                print(f"{prefix}{key}:")
                                log_attribute_recursively(value, indent + 1, visited)
                except:
                    pass
    except:
        pass


def log_condition_details(condition, item_guid: int):
    """Log non-default values from a condition for debugging."""
    print(f"\n=== Unhandled Condition for Item {item_guid} ===")
    
    # Get condition template name
    try:
        if hasattr(condition, 'template'):
            print(f"Template: {condition.template.name}")
    except:
        pass
    
    # Check base Condition attributes
    if hasattr(condition, 'Condition'):
        cond = condition.Condition
        print(f"\nCondition:")
        log_attribute_recursively(cond, indent=1)
    
    # Check all possible condition subtypes
    condition_types = [
        'ConditionDominantPatron',
        'ConditionNeedAttributeCounter',
        'ConditionPlayerCounter',
        'ConditionLocationFilter',
        'ConditionCounterProps',
        'ConditionPropsComparable',
        'ConditionAlwaysTrue',
        'ConditionObjectCount',
        'ConditionActiveEmperor',
        'ConditionReligion',
        'ConditionMonumentEventsActive',
        'ConditionEmperorRelation',
        'ConditionDiplomacyState',
        'ConditionItemUsed',
        'ConditionWarState',
        'ConditionInStorage',
        'ConditionTradeRouteCount',
        'ConditionRangeCheckProps'
    ]
    
    for cond_type in condition_types:
        if hasattr(condition, cond_type):
            cond_obj = getattr(condition, cond_type)
            print(f"\n{cond_type}:")
            log_attribute_recursively(cond_obj, indent=1)
            
            # For ObjectFilter, print it separately
            if cond_type == 'ConditionObjectCount' and hasattr(condition, 'ObjectFilter'):
                obj_filter = condition.ObjectFilter
                print(f"\nObjectFilter:")
                log_attribute_recursively(obj_filter, indent=1)


def extract_boost_condition(item_asset: Asset) -> str:
    """
    Extract the boost condition from an ItemWithBoost asset.
    
    Returns a human-readable description of when the boost activates.
    """
    try:
        condition_attr = item_asset.find("ItemWithBoost.BoostCondition.PreConditionList.Condition")
        if not condition_attr:
            return ""
        
        # Check the template name of the condition to determine its type
        condition = condition_attr
        if not condition:
            return ""
        
        # ConditionAlwaysTrue (no condition needed)
        try:
            if hasattr(condition, 'ConditionAlwaysTrue'):
                # Check if there are other conditions
                has_other_conditions = any(
                    hasattr(condition, cond_type) for cond_type in [
                        'ConditionObjectCount', 'ConditionDominantPatron', 'ConditionNeedAttributeCounter',
                        'ConditionPlayerCounter', 'ConditionActiveEmperor', 'ConditionReligion',
                        'ConditionMonumentEventsActive', 'ConditionEmperorRelation', 'ConditionDiplomacyState',
                        'ConditionItemUsed', 'ConditionWarState', 'ConditionInStorage', 'ConditionTradeRouteCount'
                    ]
                )
                if not has_other_conditions:
                    return "Always active"
        except:
            pass
        
        # ConditionObjectCount (building/object count)
        try:
            if hasattr(condition, 'ConditionObjectCount'):
                amount = condition.ConditionObjectCount.Amount()
                comparison_op = condition.ConditionObjectCount.ComparisonOp()
                
                # Map comparison operators
                comparison_map = {
                    0: ">=",
                    "AtLeast": ">=",
                    "AtMost": "<=",
                    "LessThan": "<",
                    "GreaterThan": ">",
                    "Equal": "="
                }
                op_symbol = comparison_map.get(comparison_op, ">=")
                
                # Get the object being counted
                obj_guid = condition.ObjectFilter.ObjectGUID()
                if obj_guid:
                    obj_name = get_english_name(obj_guid)
                    amount_str = str(int(amount)) if amount == int(amount) else str(amount)
                    return f"{obj_name} {op_symbol} {amount_str}"
        except:
            pass
        
        # ConditionDominantPatron
        try:
            patron_guid = condition.ConditionDominantPatron.PatronGUID()
            if patron_guid:
                patron_name = get_english_name(patron_guid)
                return f"Patron: {patron_name}"
        except:
            pass
        
        # ConditionReligion (patron requirement)
        try:
            if hasattr(condition, 'ConditionReligion'):
                religion_asset = condition.ConditionReligion.ReligionAsset()
                if religion_asset:
                    religion_name = get_english_name(religion_asset)
                    return f"Patron: {religion_name}"
        except:
            pass
        
        # ConditionNeedAttributeCounter
        try:
            need_type = condition.ConditionNeedAttributeCounter.NeedAttributeType()
            amount = condition.ConditionNeedAttributeCounter.NeedAttributeAmount()
            if need_type and amount:
                return f"{need_type} >= {amount}"
        except:
            pass
        
        # ConditionPlayerCounter
        try:
            player_counter = condition.ConditionPlayerCounter.PlayerCounter()
            comparison_op = condition.ConditionPlayerCounter.ComparisonOp()
            counter_amount = condition.ConditionPlayerCounter.CounterAmount()
            
            # Map comparison operators to symbols
            comparison_map = {
                0: ">=",  # AtLeast (default)
                "AtLeast": ">=",
                "AtMost": "<=",
                "LessThan": "<",
                "GreaterThan": ">",
                "Equal": "="
            }
            
            op_symbol = comparison_map.get(comparison_op, ">=")
            
            # Check if this is a building count condition
            context_building = condition.ConditionPlayerCounter.Context()
            if context_building:
                building_name = get_english_name(context_building)
                
                # Format the amount
                if counter_amount == int(counter_amount):
                    amount_str = str(int(counter_amount))
                else:
                    amount_str = str(counter_amount)
                
                return f"{building_name} {op_symbol} {amount_str}"
            
            # Check if this is a named counter (like NavalStrength)
            if player_counter and player_counter != 0:
                counter_name = str(player_counter)
                
                # Format the amount
                if counter_amount == int(counter_amount):
                    amount_str = str(int(counter_amount))
                else:
                    amount_str = str(counter_amount)
                
                return f"{counter_name} {op_symbol} {amount_str}"
        except:
            pass
        
        # ConditionActiveEmperor
        try:
            if hasattr(condition, 'ConditionActiveEmperor'):
                emperor = condition.ConditionActiveEmperor.EmperorParticipant()
                if emperor:
                    emperor_name = get_english_name(emperor)
                    return f"Emperor: {emperor_name}"
        except:
            pass
        
        # ConditionDiplomacyState
        try:
            if hasattr(condition, 'ConditionDiplomacyState'):
                profile2 = condition.ConditionDiplomacyState.Profile2()
                desired_state = condition.ConditionDiplomacyState.DesiredState()
                if profile2 and desired_state:
                    faction_name = get_english_name(profile2)
                    return f"Diplomacy with {faction_name}: {desired_state}"
        except:
            pass
        
        # ConditionTradeRouteCount
        try:
            if hasattr(condition, 'ConditionTradeRouteCount'):
                count = condition.ConditionTradeRouteCount.TradeRouteCount()
                if count:
                    count_str = str(int(count)) if count == int(count) else str(count)
                    return f"Trade routes >= {count_str}"
        except:
            pass
        
        # ConditionItemUsed
        try:
            if hasattr(condition, 'ConditionItemUsed'):
                item_amount = condition.ConditionItemUsed.ItemAmount()
                if item_amount:
                    amount_str = str(int(item_amount)) if item_amount == int(item_amount) else str(item_amount)
                    return f"{amount_str} items equipped"
        except:
            pass
        
        # ConditionMonumentEventsActive
        try:
            if hasattr(condition, 'ConditionMonumentEventsActive'):
                return "Monument events active"
        except:
            pass
        
        # ConditionEmperorRelation
        try:
            if hasattr(condition, 'ConditionEmperorRelation'):
                return "Emperor relation required"
        except:
            pass
        
        # ConditionWarState
        try:
            if hasattr(condition, 'ConditionWarState'):
                return "At war"
        except:
            pass
        
        # ConditionInStorage
        try:
            if hasattr(condition, 'ConditionInStorage'):
                return "Items in storage"
        except:
            pass
        
        # If we couldn't parse it, log the details and return generic message
        log_condition_details(condition, item_asset.guid)
        return "Boost condition active"
    except:
        return ""


def extract_boost_buffs(item_asset: Asset, visited_effects: set[int] = None) -> str:
    """
    Extract and format boost buff attributes from an ItemWithBoost asset.
    
    Returns formatted buff attributes similar to regular buffs.
    """
    if visited_effects is None:
        visited_effects = set()
    
    try:
        boost_buffs_attr = item_asset.find("ItemWithBoost.BoostBuffs")
        if not boost_buffs_attr:
            return ""
        
        buff_descriptions = []
        for buff_entry in boost_buffs_attr:
            try:
                buff_guid = buff_entry.GUID.guid
                if buff_guid:
                    buff_asset = assets[buff_guid]
                    if buff_asset:
                        buff_desc = format_buff_attributes(buff_asset, visited_effects)
                        if buff_desc and buff_desc != "No attributes":
                            buff_descriptions.append(buff_desc)
            except:
                pass
        
        return " | ".join(buff_descriptions) if buff_descriptions else ""
    except:
        return ""


# === Ship Buff Extractors ===

def extract_movement_upgrades(buff_asset: Asset) -> list[str]:
    """Extract movement-related upgrade attributes from a ShipBuff."""
    attributes = []
    
    try:
        attr = buff_asset.find("MovementUpgrade.BuffBaseSpeedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Movement Speed: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("MovementUpgrade.BuffReduceCargoImpactUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Reduce Cargo Impact: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("MovementUpgrade.BuffReduceDamageImpactUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Reduce Damage Impact: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("MovementUpgrade.BuffReduceNegativeWindImpactUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Reduce Negative Wind Impact: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("MovementUpgrade.BuffReducePositiveWindImpactUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Reduce Positive Wind Impact: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("MovementUpgrade.BuffFavorableWindAngle")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Favorable Wind Angle: {sign}{attr()}°")
    except:
        pass
    
    try:
        attr = buff_asset.find("MovementUpgrade.BuffTransferSpeedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Transfer Speed: {sign}{attr()}%")
    except:
        pass
    
    return attributes


def extract_health_upgrades(buff_asset: Asset) -> list[str]:
    """Extract health-related upgrade attributes from a ShipBuff."""
    attributes = []
    
    try:
        attr = buff_asset.find("HealthUpgrade.BaseHealthUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Hitpoints: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("HealthUpgrade.SelfHealUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Self Heal: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("HealthUpgrade.SelfHealPausedTimeIfAttackedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Self Heal Pause Time: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("HealthUpgrade.PassiveRuinRepairSpeedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Ruin Repair Speed: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("HealthUpgrade.EncampedUnitSelfHealMultiplierUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Encamped Self Heal: {sign}{attr()}%")
    except:
        pass
    
    return attributes


def extract_vehicle_upgrades(buff_asset: Asset) -> list[str]:
    """Extract vehicle-related upgrade attributes from a ShipBuff."""
    attributes = []
    
    try:
        if buff_asset.find("VehicleUpgrade.ActivateWhiteFlag")():
            attributes.append("White Flag Active")
    except:
        pass
    
    try:
        if buff_asset.find("VehicleUpgrade.ActivatePirateFlag")():
            attributes.append("Pirate Flag Active")
    except:
        pass
    
    return attributes


def extract_trade_ship_upgrades(buff_asset: Asset) -> list[str]:
    """Extract trade ship upgrade attributes from a ShipBuff."""
    attributes = []
    
    try:
        attr = buff_asset.find("TradeShipUpgrade.ActiveTradePriceInPercent")
        if attr and attr() != 100:
            discount = 100 - attr()
            sign = "-" if discount > 0 else "+"
            attributes.append(f"Trade Price: {sign}{abs(discount)}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("TradeShipUpgrade.LoadingSpeedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Loading Speed: {sign}{attr()}%")
    except:
        pass
    
    return attributes


def extract_unit_upgrades(buff_asset: Asset) -> list[str]:
    """Extract unit upgrade attributes from a ShipBuff."""
    attributes = []
    
    try:
        attr = buff_asset.find("UnitUpgrade.DiscoveryRadiusUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Discovery Radius: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("UnitUpgrade.RewardMoneyPerDestroyedBuildingUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Reward per Building: {sign}{int(value)}")
            else:
                attributes.append(f"Reward per Building: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("UnitUpgrade.RewardMoneyPerDestroyedShipUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Reward per Ship: {sign}{int(value)}")
            else:
                attributes.append(f"Reward per Ship: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("UnitUpgrade.DefenseUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Defense: {sign}{int(value)}")
            else:
                attributes.append(f"Defense: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("UnitUpgrade.ArmorUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Armor: {sign}{int(value)}")
            else:
                attributes.append(f"Armor: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("UnitUpgrade.ShieldUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Shield: {sign}{int(value)}")
            else:
                attributes.append(f"Shield: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("UnitUpgrade.AccuracyUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Accuracy: {sign}{int(value)}%")
            else:
                attributes.append(f"Accuracy: {sign}{value}%")
    except:
        pass
    
      # Module Accuracy Upgrades
    try:
        attr = buff_asset.find("UnitUpgrade.AccuracyArcherModuleUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Archer Module Accuracy: {sign}{int(value)}%")
            else:
                attributes.append(f"Archer Module Accuracy: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.AccuracyCatapultModuleUpgrage")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Catapult Module Accuracy: {sign}{int(value)}%")
            else:
                attributes.append(f"Catapult Module Accuracy: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.AccuracyBallistaModuleUpgrage")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Ballista Module Accuracy: {sign}{int(value)}%")
            else:
                attributes.append(f"Ballista Module Accuracy: {sign}{value}%")
    except:
        pass

    # Attack Range Upgrades
    try:
        attr = buff_asset.find("UnitUpgrade.DistanceAttackRangePercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Attack Range: {sign}{int(value)}%")
            else:
                attributes.append(f"Attack Range: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.DistanceAttackRangeArcherModulePercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Archer Module Attack Range: {sign}{int(value)}%")
            else:
                attributes.append(f"Archer Module Attack Range: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.DistanceAttackRangeCatapultModulePercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Catapult Module Attack Range: {sign}{int(value)}%")
            else:
                attributes.append(f"Catapult Module Attack Range: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.DistanceAttackRangeBallistaModulePercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Ballista Module Attack Range: {sign}{int(value)}%")
            else:
                attributes.append(f"Ballista Module Attack Range: {sign}{value}%")
    except:
        pass

    # Unit Offense Upgrades
    try:
        attr = buff_asset.find("UnitUpgrade.OffenseMeleeUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Melee Offense: {sign}{int(value)}")
            else:
                attributes.append(f"Melee Offense: {sign}{value}")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.OffenseChargeUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Charge Offense: {sign}{int(value)}")
            else:
                attributes.append(f"Charge Offense: {sign}{value}")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.OffenseRangedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Ranged Offense: {sign}{int(value)}")
            else:
                attributes.append(f"Ranged Offense: {sign}{value}")
    except:
        pass

    # Module Offense Upgrades
    try:
        attr = buff_asset.find("UnitUpgrade.OffenseArcherModuleRangedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Archer Module Offense: {sign}{int(value)}")
            else:
                attributes.append(f"Archer Module Offense: {sign}{value}")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.OffenseCatapultModuleRangedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Catapult Module Offense: {sign}{int(value)}")
            else:
                attributes.append(f"Catapult Module Offense: {sign}{value}")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.OffenseBallistaModuleRangedUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Ballista Module Offense: {sign}{int(value)}")
            else:
                attributes.append(f"Ballista Module Offense: {sign}{value}")
    except:
        pass

    # Attack Speed Upgrades
    try:
        attr = buff_asset.find("UnitUpgrade.AttackSpeedArcherModulePercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Archer Module Attack Speed: {sign}{int(value)}%")
            else:
                attributes.append(f"Archer Module Attack Speed: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.AttackSpeedCatapultModulePercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Catapult Module Attack Speed: {sign}{int(value)}%")
            else:
                attributes.append(f"Catapult Module Attack Speed: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.AttackSpeedBallistaModulePercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Ballista Module Attack Speed: {sign}{int(value)}%")
            else:
                attributes.append(f"Ballista Module Attack Speed: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.AttackSpeedTorchPercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Torch Attack Speed: {sign}{int(value)}%")
            else:
                attributes.append(f"Torch Attack Speed: {sign}{value}%")
    except:
        pass

    try:
        attr = buff_asset.find("UnitUpgrade.AttackSpeedRangedPercentualUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Ranged Attack Speed: {sign}{int(value)}%")
            else:
                attributes.append(f"Ranged Attack Speed: {sign}{value}%")
    except:
        pass

    # Morale Upgrade
    try:
        attr = buff_asset.find("UnitUpgrade.MaximumMoraleUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Maximum Morale: {sign}{int(value)}")
            else:
                attributes.append(f"Maximum Morale: {sign}{value}")
    except:
        pass
    
    return attributes


def extract_item_container_upgrades(buff_asset: Asset) -> list[str]:
    """Extract item container upgrade attributes from a ShipBuff."""
    attributes = []
    
    try:
        attr = buff_asset.find("ItemContainerUpgrade.SocketCountUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Socket Count: {sign}{int(value)}")
            else:
                attributes.append(f"Socket Count: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("ItemContainerUpgrade.SlotCountUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Cargo Slots: {sign}{int(value)}")
            else:
                attributes.append(f"Cargo Slots: {sign}{value}")
    except:
        pass
    
    return attributes


def extract_sellable_upgrades(buff_asset: Asset) -> list[str]:
    """Extract sellable upgrade attributes from a ShipBuff."""
    attributes = []
    
    try:
        attr = buff_asset.find("SellableUpgrade.SellPriceFactorUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Sell Price: {sign}{attr()}%")
    except:
        pass
    
    return attributes


def extract_ship_buff_attributes(buff_asset: Asset) -> list[str]:
    """Extract all ship-specific buff attributes."""
    attributes = []
    attributes.extend(extract_movement_upgrades(buff_asset))
    attributes.extend(extract_health_upgrades(buff_asset))
    attributes.extend(extract_vehicle_upgrades(buff_asset))
    attributes.extend(extract_trade_ship_upgrades(buff_asset))
    attributes.extend(extract_unit_upgrades(buff_asset))
    attributes.extend(extract_item_container_upgrades(buff_asset))
    attributes.extend(extract_sellable_upgrades(buff_asset))
    return attributes


# === Building Buff Extractors ===

def extract_functional_effects(buff_asset: Asset, visited_effects: set[int]) -> list[str]:
    """Extract attributes from AdditionalFunctionalEffect recursively."""
    attributes = []
    
    try:
        attr = buff_asset.find("BuildingUpgrade.AdditionalFunctionalEffect")
        if attr and attr():
            effect = attr()
            if effect.guid not in visited_effects:
                visited_effects.add(effect.guid)
                try:
                    effect_cfg = effect.Effect
                    if effect_cfg:
                        for buff in effect_cfg.Buffs:
                            if buff.GUID.guid:
                                buff_asset_nested = assets[buff.GUID.guid]
                                if buff_asset_nested:
                                    nested_attrs = format_buff_attributes(buff_asset_nested, visited_effects)
                                    if nested_attrs and nested_attrs != "No attributes":
                                        attributes.append(f"[within range of targets] {nested_attrs}")
                except:
                    pass
    except:
        pass
    
    return attributes


def extract_building_upgrades(buff_asset: Asset) -> list[str]:
    """Extract BuildingUpgrade attributes."""
    attributes = []
    
    try:
        attr = buff_asset.find("BuildingUpgrade.AdditionalAttributes")
        if attr:
            for need_attr in attr:
                try:
                    amount = need_attr.AmountOrPercent()
                    if amount and amount != 0:
                        sign = "+" if amount > 0 else ""
                        attributes.append(f"{need_attr.name}: {sign}{amount}")
                except:
                    pass
    except:
        pass
    
    try:
        attr = buff_asset.find("BuildingUpgrade.AttributeModifierInPercent")
        if attr and attr() != 0:
            attributes.append(f"Attribute Modifier: {attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("BuildingUpgrade.WorkforceModifierInPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Workforce Output: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("BuildingUpgrade.AdditionalWorkforces")
        if attr and len(attr._value_list) > 0:
            workforce_names = []
            for wf in attr:
                try:
                    wf_asset = wf.WorkforceGUID()
                    if wf_asset:
                        workforce_names.append(get_english_name(wf_asset))
                except:
                    pass
            if workforce_names:
                attributes.append(f"Additional Workforces: {', '.join(workforce_names)}")
    except:
        pass
    
    return attributes


def extract_residence_upgrades(buff_asset: Asset) -> list[str]:
    """Extract ResidenceUpgrade attributes."""
    attributes = []
    
    try:
        attr = buff_asset.find("ResidenceUpgrade.ProvidedNeedUpgrade")
        if attr and len(attr._value_list) > 0:
            needs = []
            for need in attr:
                try:
                    need_asset = need.ProvidedNeed()
                    if need_asset:
                        needs.append(get_english_name(need_asset))
                except:
                    pass
            if needs:
                attributes.append(f"Provided Needs: {', '.join(needs)}")
    except:
        pass
    
    try:
        attr = buff_asset.find("ResidenceUpgrade.GoodConsumptionUpgrade")
        if attr and len(attr._value_list) > 0:
            for upgrade in attr:
                try:
                    need = upgrade.ProvidedNeed()
                    amount = upgrade.AmountInPercent()
                    if need and amount:
                        attributes.append(f"{get_english_name(need)} Consumption: {amount}%")
                except:
                    pass
    except:
        pass
    
    try:
        attr = buff_asset.find("ResidenceUpgrade.ConsumptionModifierInPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Consumption Rate: {sign}{attr()}%")
    except:
        pass
    
    # NeedProvidedNeedAttributes - conditional bonuses
    try:
        attr = buff_asset.find("ResidenceUpgrade.NeedProvidedNeedAttributes")
        if attr:
            change_needs = attr.ChangeNeedAttributesOf
            if change_needs and len(change_needs._value_list) > 0:
                provided_products = []
                for item in change_needs:
                    try:
                        provided_product = item.ProvidedProduct()
                        if provided_product:
                            provided_products.append(get_english_name(provided_product))
                    except:
                        pass
                
                additional_attrs = attr.AdditionalNeedAttributes
                if additional_attrs:
                    attr_list = []
                    for need_attr in additional_attrs:
                        try:
                            amount = need_attr.AmountOrPercent()
                            if amount and amount != 0:
                                sign = "+" if amount > 0 else ""
                                attr_list.append(f"{need_attr.name}: {sign}{amount}")
                        except:
                            pass
                    
                    if attr_list and provided_products:
                        condition = " or ".join(provided_products)
                        attrs_str = ", ".join(attr_list)
                        attributes.append(f"When {condition} provided: {attrs_str}")
    except:
        pass
    
    return attributes


def extract_factory_upgrades(buff_asset: Asset) -> list[str]:
    """Extract FactoryUpgrade attributes."""
    attributes = []
    
    try:
        attr = buff_asset.find("FactoryUpgrade.ProductivityUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Productivity: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.AddedFertility")
        if attr and attr():
            fertility = attr()
            attributes.append(f"Added Fertility: {get_english_name(fertility)}")
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.FertilityPercent")
        if attr and attr() != 100:
            attributes.append(f"Fertility Percent: {attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.AdditionalOutput")
        if attr and len(attr._value_list) > 0:
            outputs = []
            for output in attr:
                try:
                    product = output.Product()
                    amount = output.Amount()
                    cycle = output.AdditionalOutputCycle()
                    if product and amount:
                        prod_name = get_english_name(product)
                        if cycle > 1:
                            outputs.append(f"+{amount} {prod_name} (every {cycle} cycles)")
                        else:
                            outputs.append(f"+{amount} {prod_name}")
                except:
                    pass
            if outputs:
                attributes.append(f"Additional Output: {', '.join(outputs)}")
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.InputAmountUpgrade")
        if attr and len(attr._value_list) > 0:
            for upgrade in attr:
                try:
                    product = upgrade.Product()
                    amount = upgrade.Amount()
                    if product and amount:
                        sign = "+" if amount > 0 else ""
                        attributes.append(f"{get_english_name(product)} Input: {sign}{amount}")
                except:
                    pass
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.ReplaceInputs")
        if attr and len(attr._value_list) > 0:
            for replacement in attr:
                try:
                    old = replacement.OldInput()
                    new = replacement.NewInput()
                    if old and new:
                        attributes.append(f"Replace Input: {get_english_name(old)} → {get_english_name(new)}")
                except:
                    pass
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.NeededAreaUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Needed Area: {sign}{attr()}%")
    except:
        pass
    
    try:
        if buff_asset.find("FactoryUpgrade.CanUseMarsh")():
            attributes.append("Can Use Marsh")
    except:
        pass
    
    try:
        if buff_asset.find("FactoryUpgrade.CanUseForest")():
            attributes.append("Can Use Forest")
    except:
        pass
    
    try:
        if buff_asset.find("FactoryUpgrade.CanUseMeadow")():
            attributes.append("Can Use Meadow")
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.FuelDurationPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Fuel Duration: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("FactoryUpgrade.InfluenceRadiusUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Influence Radius: {sign}{attr()}%")
    except:
        pass
    
    return attributes


def extract_maintenance_upgrades(buff_asset: Asset) -> list[str]:
    """Extract MaintenanceUpgrade attributes."""
    attributes = []
    
    try:
        attr = buff_asset.find("MaintenanceUpgrade.MaintenanceFactorUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Maintenance Cost: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("MaintenanceUpgrade.WorkforceMaintenanceFactorUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Workforce Maintenance: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("MaintenanceUpgrade.ReplaceWorkforce")
        if attr and hasattr(attr, "OldWorkforce"):
            old_wf = attr.OldWorkforce()
            new_wf = attr.NewWorkforce()
            if old_wf and new_wf:
                attributes.append(f"Replace Workforce: {get_english_name(old_wf)} → {get_english_name(new_wf)}")
    except:
        pass
    
    try:
        attr = buff_asset.find("MaintenanceUpgrade.EncampedUnitScalingFactorUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Encamped Unit Scaling: {sign}{attr()}")
    except:
        pass
    
    return attributes


def extract_other_building_upgrades(buff_asset: Asset) -> list[str]:
    """Extract miscellaneous building upgrade attributes."""
    attributes = []
    
    # ModuleOwnerUpgrade
    try:
        attr = buff_asset.find("ModuleOwnerUpgrade.ModuleLimitPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Module Limit: {sign}{attr()}%")
    except:
        pass
    
    # CityInstitutionUpgrade
    try:
        attr = buff_asset.find("CityInstitutionUpgrade.ResolverUnitCountUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Resolver Unit Count: {sign}{int(value)}")
            else:
                attributes.append(f"Resolver Unit Count: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("CityInstitutionUpgrade.ResolverResolveDurationUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Resolver Resolve Duration: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("CityInstitutionUpgrade.ResolverRepairDurationUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Resolver Repair Duration: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("CityInstitutionUpgrade.ResolverRangeUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Resolver Range: {sign}{attr()}%")
    except:
        pass
    
    # IncidentInfectableUpgrade
    try:
        attr = buff_asset.find("IncidentInfectableUpgrade.ResistanceThresholdUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Resistance Threshold: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("IncidentInfectableUpgrade.IncidentImmunity")
        if attr and attr():
            immunities = attr()
            if immunities:
                attributes.append(f"Incident Immunity: {', '.join(immunities)}")
    except:
        pass
    
    # RecruitmentUpgrade
    try:
        attr = buff_asset.find("RecruitmentUpgrade.ConstructionCostInPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Construction Cost: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("RecruitmentUpgrade.ConstructionSpeedInPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Construction Speed: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("RecruitmentUpgrade.RecruitmentCostInPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Recruitment Cost: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("RecruitmentUpgrade.RecruitmentSpeedInPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Recruitment Speed: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("RecruitmentUpgrade.OnlyBuffSingleAssemblyOption")
        if attr and attr():
            option = attr()
            attributes.append(f"Only Buff: {get_english_name(option)}")
    except:
        pass
    
    # AqueductUpgrade
    try:
        attr = buff_asset.find("AqueductUpgrade.AqueductConsumedWaterUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Aqueduct Consumed Water: {sign}{attr()}%")
    except:
        pass
    
    try:
        attr = buff_asset.find("AqueductUpgrade.AqueductWaterSupplyUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Aqueduct Water Supply: {sign}{attr()}%")
    except:
        pass
    
    # WarehouseUpgrade
    try:
        attr = buff_asset.find("WarehouseUpgrade.StorageCapacityModifier")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            value = attr()
            if value == int(value):
                attributes.append(f"Storage Capacity: {sign}{int(value)}")
            else:
                attributes.append(f"Storage Capacity: {sign}{value}")
    except:
        pass
    
    try:
        attr = buff_asset.find("WarehouseUpgrade.AdditionalLoadingSpeedInPercent")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Loading Speed: {sign}{attr()}%")
    except:
        pass
    
    # IrrigationUpgrade
    try:
        attr = buff_asset.find("IrrigationUpgrade.PipeCapacityUpgrade")
        if attr and attr() != 0:
            sign = "+" if attr() > 0 else ""
            attributes.append(f"Pipe Capacity: {sign}{attr()}%")
    except:
        pass
    
    return attributes


def extract_building_buff_attributes(buff_asset: Asset, visited_effects: set[int]) -> list[str]:
    """Extract all building-specific buff attributes."""
    attributes = []
    attributes.extend(extract_functional_effects(buff_asset, visited_effects))
    attributes.extend(extract_building_upgrades(buff_asset))
    attributes.extend(extract_residence_upgrades(buff_asset))
    attributes.extend(extract_factory_upgrades(buff_asset))
    attributes.extend(extract_maintenance_upgrades(buff_asset))
    attributes.extend(extract_other_building_upgrades(buff_asset))
    return attributes


# === Main Orchestrator ===

def format_buff_attributes(buff_asset: Asset, visited_effects: set[int] = None) -> str:
    """
    Extract and format ALL attributes from a BuildingBuff or ShipBuff asset.
    
    This is the main entry point that delegates to specialized extractors.
    """
    if visited_effects is None:
        visited_effects = set()
    
    attributes = []
    
    # Route to appropriate extractor based on buff type
    if "ShipBuff" in buff_asset.template.name:
        attributes.extend(extract_ship_buff_attributes(buff_asset))
    else:
        attributes.extend(extract_building_buff_attributes(buff_asset, visited_effects))
    
    return "; ".join(attributes) if attributes else "No attributes"

Indexed 503 items in reward pools


In [116]:
def extract_all_items(assets: AssetCache, templates: t.Any, item_to_pools: dict[int, set[int]]) -> list[dict[str, str]]:
    """
    Extract all items from the asset cache.
    
    Returns a list of dictionaries with keys: guid, name, rarity, trade_price, targets, buffs, boost_condition, boost_buffs, source
    """
    items_data = []

    for template_name in ["Item", "ItemWithBoost"]:
        if template_name not in templates:
            print(f"Warning: Template '{template_name}' not found")
            continue
        
        for asset in templates[template_name].assets:
            try:
                item = {
                    "guid": asset.guid,
                    "name": get_english_name(asset),
                    "rarity": "",
                    "trade_price": "",
                    "targets": "",
                    "buffs": "",
                    "boost_condition": "",
                    "boost_buffs": "",
                    "source": ""
                }
                
                # Get rarity
                try:
                    rarity = asset.Item.Rarity()
                    if rarity:
                        item["rarity"] = rarity
                except:
                    pass
                
                # Get trade price
                try:
                    trade_price = asset.Item.TradePrice()
                    if trade_price:
                        item["trade_price"] = str(trade_price)
                except:
                    pass
                
                # Get targets from Effect property and convert to names
                try:
                    effect = asset.Effect
                    if effect:
                        target_guids = []
                        for pool in effect.Targets:
                            target_guids.extend(flatten_pool(pool.GUID()))
                        if target_guids:
                            item["targets"] = get_target_names(target_guids)
                except:
                    pass
                
                # Get buffs from Effect property and extract their attributes
                try:
                    effect = asset.Effect
                    if effect:
                        buff_descriptions = []
                        for buff in effect.Buffs:
                            if buff.GUID.guid:
                                buff_asset = assets[buff.GUID.guid]
                                if buff_asset:
                                    buff_desc = format_buff_attributes(buff_asset)
                                    if buff_desc and buff_desc != "No attributes":
                                        buff_descriptions.append(buff_desc)
                        if buff_descriptions:
                            item["buffs"] = " | ".join(buff_descriptions)
                except:
                    pass
                
                # Get boost condition and boost buffs for ItemWithBoost items
                if "ItemWithBoost" in asset.template.name:
                    try:
                        boost_condition = extract_boost_condition(asset)
                        if boost_condition:
                            item["boost_condition"] = boost_condition
                    except:
                        pass
                    
                    try:
                        boost_buffs = extract_boost_buffs(asset)
                        if boost_buffs:
                            item["boost_buffs"] = boost_buffs
                    except:
                        pass
                
                # Find sources where this item can be obtained
                try:
                    sources = find_item_sources(asset.guid, item_to_pools, assets)
                    if sources:
                        item["source"] = "; ".join(sources)
                except Exception as e:
                    # Source finding is best-effort, don't fail extraction if it errors
                    pass
                
                items_data.append(item)
                
            except Exception as e:
                print(f"Error processing asset {asset.guid}: {e}")
                continue

    print(f"Extracted {len(items_data)} items")
    return items_data


# Extract all items
items_data = extract_all_items(assets, templates, item_to_pools)


=== Unhandled Condition for Item 80179 ===

Condition:
  SubConditionCompletionOrder: Parallel
  CountForAchievementProgress: True

ConditionLocationFilter:
  LocationFilter:
    LocationFilter: LocationFilter

ConditionAlwaysTrue:

ConditionReligion:
Extracted 391 items


In [117]:
resolve_reward_pool_chain(71438, item_to_pools, assets)  # Example item GUID for testing

set()

In [118]:
def preview_items(items_data: list[dict[str, str]], num_items: int = 10) -> None:
    """Display a preview of the first N items in a formatted table."""
    df = pd.DataFrame(items_data)
    print(f"Total items: {len(df)}")
    print(f"\nColumns: {', '.join(df.columns)}")
    print(f"\nFirst {num_items} items:")
    display(df.head(num_items))


# Preview the extracted items
preview_items(items_data)

Total items: 391

Columns: guid, name, rarity, trade_price, targets, buffs, boost_condition, boost_buffs, source

First 10 items:


,guid,name,rarity,trade_price,targets,buffs,boost_condition,boost_buffs,source
0,71438,"Gaius Julius Lupus, Castor of Fortunes",Epic,100000,"Market, Market",[within range of targets] Money: +1.5,,,
1,71441,"Brutus Julius Lupus, Pollux of Polities",Epic,100000,"Libertus Residence, Plebeian Residence, Eques ...",Workforce Output: +20.0%,,,
2,42057,"Abdfil, Elephant Handler",Epic,100000,"Libertus Residence, Plebeian Residence, Eques ...",Prestige: +1.5,,,
3,44431,Elephant Handler,Rare,15000,"Wheat Farm, Wheat Farm, Flax Farm, Flax Farm, ...",Prestige: +1.0; Productivity: +20.0%,,,
4,94567,"Aodhan, Master Lutist of the Scathach",Epic,100000,"Libertus Residence, Plebeian Residence, Eques ...","When Bardic Hearth provided: Happiness: +1.0, ...",,,
5,125560,Abuccus,Rare,15000,"Watchtower, Watchtower",[within range of targets] Money: +0.5,,,
6,95403,"Servia Bellia, Lily of the Coast",Epic,100000,"Fishing Hut, Scomber's Shack, Salt Ponds, Snai...",[within range of targets] Money: +1.0; Product...,,,
7,96815,"Gigantulas, Polyphemian Captain",Epic,100000,"Quinquireme, Penteconter, Trireme, Flagship, R...",Movement Speed: -20.0%; Hitpoints: +150.0%; Se...,,,
8,96821,"Publius Quintus, Amphipraetorian",Epic,100000,"Vigiles, Vigiles, Custodia, Custodes, Medici, ...",[within range of targets] Money: +1.0; Workfor...,,,
9,96819,Virtuous Volunteer,Rare,15000,"Tiler, Tiler, Concrete Mixer, Concrete Mixer, ...",Productivity: +15.0%; Maintenance Cost: -25.0%...,,,


In [ ]:
def save_items_to_csv(items_data: list[dict[str, str]], output_path: Path) -> None:
    """Save items data to a CSV file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["guid", "name", "rarity", "trade_price", "targets", "buffs", "boost_condition", "boost_buffs", "source"])
        writer.writeheader()
        writer.writerows(items_data)
    
    print(f"Saved {len(items_data)} items to {output_path.absolute()}")


# Save to CSV
output_path = Path("results") / "tables/items.csv"
save_items_to_csv(items_data, output_path)

Saved 391 items to c:\asset-extractor\results\items.csv


In [120]:
assets.datasets["PlayerCounter"].literals

['ObjectCount',
 'TradeProductBought',
 'TradeProductSold',
 'PopulationByLevel',
 'PopulationByGroup',
 'PopulationTotal',
 'GoodsInStock',
 'ObjectDestroyed',
 'ObjectLost',
 'StreetConnection',
 'NoStreetConnection',
 'GovernorLevel',
 'BuildingsMoved',
 'IslandSettled',
 'BuildingsDemolished',
 'QuestComponentCreated',
 'QuestComponentEnded',
 'QuestObjectiveComponentSolved',
 'QuestObjectiveComponentFailed',
 'QuestObjectiveComponentAbortedManually',
 'QuestStorylineStarted',
 'BankruptCounter',
 'PopulationSatisfactionByGood',
 'InfectedObjects',
 'MoneyBalance',
 'CollectablesCollected',
 'PassiveTradeBalance',
 'ShipsSoldToParticipant',
 'ParticipantDefeated',
 'PirateDefeated',
 'QuestPoolQuestsSolved',
 'IncidentActive',
 'NavalStrength',
 'Forestation',
 'RuinCount',
 'ItemCrafted',
 'MonumentEventsFinished',
 'GamepadActionsConsumed',
 'TradeMoneyEarned',
 'TradeMoneyPayed',
 'Lifetime',
 'CityStatus',
 'ActiveEmperorReputation',
 'PatronDevotion',
 'PatronIslands',
 'ArmyS

In [121]:
def display_item_statistics(items_data: list[dict[str, str]]) -> None:
    """Display comprehensive statistics about the extracted items."""
    df = pd.DataFrame(items_data)
    
    print("\n=== Item Statistics ===")
    print(f"Total items: {len(items_data)}")

    # Count by rarity
    rarity_counts = df['rarity'].value_counts()
    print(f"\nItems by rarity:")
    for rarity, count in rarity_counts.items():
        print(f"  {rarity}: {count}")

    # Items with effects (have buffs or targets)
    items_with_buffs = df[df['buffs'] != ''].shape[0]
    items_with_targets = df[df['targets'] != ''].shape[0]
    items_with_sources = df[df['source'] != ''].shape[0]
    print(f"\nItems with buffs: {items_with_buffs}")
    print(f"Items with targets: {items_with_targets}")
    print(f"Items with known sources: {items_with_sources}")

    # Price statistics
    df_with_price = df[df['trade_price'] != '']
    if len(df_with_price) > 0:
        df_with_price['trade_price_num'] = pd.to_numeric(df_with_price['trade_price'])
        print(f"\nPrice statistics (for {len(df_with_price)} items with prices):")
        print(f"  Min: {df_with_price['trade_price_num'].min()}")
        print(f"  Max: {df_with_price['trade_price_num'].max()}")
        print(f"  Average: {df_with_price['trade_price_num'].mean():.2f}")


# Display statistics
display_item_statistics(items_data)


=== Item Statistics ===
Total items: 391

Items by rarity:
  Rare: 127
  Epic: 115
  Common: 68
  Legendary: 57
  Unique: 17
  Quest: 6
  Uncommon: 1

Items with buffs: 374
Items with targets: 382
Items with known sources: 320

Price statistics (for 391 items with prices):
  Min: 100
  Max: 600000
  Average: 116039.90
